# Spotify Playlist EDA

This notebook explores the playlist dataframe after API extraction and feature creation. The goal is to understand data quality, listening taste, language/genre patterns, and modelling-ready signals before moving into recommendation or clustering.

## 1. Setup

In [ ]:
import ast
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd

alt.data_transformers.disable_max_rows()
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

In [ ]:
def find_data_dir():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        notebook_data = candidate / "notebooks" / "data"
        local_data = candidate / "data"
        if notebook_data.exists():
            return notebook_data
        if candidate.name == "notebooks" and local_data.exists():
            return local_data
    return Path("data")


DATA_DIR = find_data_dir()
DATA_PATH = DATA_DIR / "df.csv"

df = pd.read_csv(DATA_PATH)

# Parse dates early so time-based EDA is easier.
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["added_at"] = pd.to_datetime(df["added_at"], errors="coerce", utc=True)
df["release_year"] = df["release_date"].dt.year
df["added_date"] = df["added_at"].dt.strftime("%Y-%m-%d")

df.head()

,track_id,track_name,artist_names,artist_ids,main_artist_name,main_artist_id,album_id,album_name,release_date,duration_ms,duration_min,popularity,explicit,track_url,added_at,is_local,artist_name_from_api,main_artist_genres,artist_popularity,artist_followers,title_script,song_title_language,genre_language_tags,release_year,added_date
0,0VGPId3riSeBV5FTRn850F,需要人陪,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,5lpy0FxmneKqFb4zeoiSHM,十八般武藝,2010-08-12,251146,4.185767,50,False,https://open.spotify.com/track/0VGPId3riSeBV5FTRn850F,2025-11-20 07:03:00+00:00,False,Leehom Wang,"['mandopop', 'c-pop', 'taiwanese pop', 'chinese r&b', 'malay pop']",57,1192346,chinese,Chinese-script title,['Chinese-associated'],2010.0,2025-11-20
1,6aF2RVTPJSJCM1MpUjne5X,落葉歸根,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,1lxlQW182pklrfpD93faq1,改變自己,2007-07-01,315226,5.253767,51,False,https://open.spotify.com/track/6aF2RVTPJSJCM1MpUjne5X,2025-11-20 07:03:00+00:00,False,Leehom Wang,"['mandopop', 'c-pop', 'taiwanese pop', 'chinese r&b', 'malay pop']",57,1192346,chinese,Chinese-script title,['Chinese-associated'],2007.0,2025-11-20
2,68RtalgSqaCbeZN42QeMS0,花田錯,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,34D46J9tIGCqAj3FeiEA9O,蓋世英雄,2005-12-30,227960,3.799333,50,False,https://open.spotify.com/track/68RtalgSqaCbeZN42QeMS0,2025-11-20 07:03:00+00:00,False,Leehom Wang,"['mandopop', 'c-pop', 'taiwanese pop', 'chinese r&b', 'malay pop']",57,1192346,chinese,Chinese-script title,['Chinese-associated'],2005.0,2025-11-20
3,3agtg0x11wPvLIWkYR39nZ,Somewhere I Belong,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,4Gfnly5CzMJQqkUFfoHaP3,Meteora,2003-03-25,213933,3.565550,81,False,https://open.spotify.com/track/3agtg0x11wPvLIWkYR39nZ,2025-11-20 07:03:00+00:00,False,Linkin Park,"['nu metal', 'rap metal', 'rock', 'alternative metal']",89,34542635,latin,Latin-script title,['Western-associated'],2003.0,2025-11-20
4,57BrRMwf9LrcmuOsyGilwr,Crawling,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,6hPkbAV3ZXpGZBGUvL6jVM,Hybrid Theory (Bonus Edition),2000-10-24,208960,3.482667,84,False,https://open.spotify.com/track/57BrRMwf9LrcmuOsyGilwr,2025-11-20 07:03:00+00:00,False,Linkin Park,"['nu metal', 'rap metal', 'rock', 'alternative metal']",89,34542635,latin,Latin-script title,['Western-associated'],2000.0,2025-11-20


## 2. Data Quality Check

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Duplicate track IDs: {df['track_id'].duplicated().sum():,}")
print(f"Date range added: {df['added_at'].min()} to {df['added_at'].max()}")
print(f"Release year range: {int(df['release_year'].min())} to {int(df['release_year'].max())}")

df.dtypes.to_frame("dtype")

Rows: 238
Columns: 25
Duplicate track IDs: 0
Date range added: 2025-11-20 07:03:00+00:00 to 2026-05-20 20:59:55+00:00
Release year range: 1997 to 2026


,dtype
track_id,object
track_name,object
artist_names,object
artist_ids,object
main_artist_name,object
main_artist_id,object
album_id,object
album_name,object
release_date,datetime64[ns]
duration_ms,int64


In [ ]:
missing_summary = (
    df.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_pct=lambda x: x["missing_count"] / len(df) * 100)
    .query("missing_count > 0")
    .sort_values("missing_pct", ascending=False)
)

missing_summary

,missing_count,missing_pct
release_date,6,2.521008
release_year,6,2.521008


In [ ]:
# Useful sanity checks for modelling and visualisation.
quality_checks = pd.Series({
    "tracks_with_empty_genre_list": (df["main_artist_genres"].fillna("[]") == "[]").sum(),
    "tracks_with_unknown_title_language": df["song_title_language"].isna().sum(),
    "tracks_with_zero_popularity": (df["popularity"] == 0).sum(),
    "local_tracks": df["is_local"].sum(),
    "explicit_tracks": df["explicit"].sum(),
})

quality_checks.to_frame("count")

,count
tracks_with_empty_genre_list,68
tracks_with_unknown_title_language,0
tracks_with_zero_popularity,5
local_tracks,0
explicit_tracks,11


## 3. Overall Playlist Profile

In [ ]:
summary = pd.Series({
    "tracks": df["track_id"].nunique(),
    "main_artists": df["main_artist_id"].nunique(),
    "albums": df["album_id"].nunique(),
    "avg_track_duration_min": df["duration_min"].mean(),
    "median_track_duration_min": df["duration_min"].median(),
    "avg_track_popularity": df["popularity"].mean(),
    "median_track_popularity": df["popularity"].median(),
    "avg_artist_popularity": df["artist_popularity"].mean(),
    "median_artist_followers": df["artist_followers"].median(),
})

summary.round(2).to_frame("value")

,value
tracks,238.00
main_artists,134.00
albums,208.00
avg_track_duration_min,3.79
median_track_duration_min,3.76
avg_track_popularity,49.68
median_track_popularity,50.50
avg_artist_popularity,61.37
median_artist_followers,1270568.00


In [ ]:
numeric_cols = ["duration_min", "popularity", "artist_popularity", "artist_followers", "release_year"]
df[numeric_cols].describe().round(2)

,duration_min,popularity,artist_popularity,artist_followers,release_year
count,238.00,238.00,238.00,2.380000e+02,232.00
mean,3.79,49.68,61.37,1.072612e+07,2011.65
std,0.64,22.04,19.68,1.958973e+07,6.64
min,2.06,0.00,12.00,2.580000e+02,1997.00
25%,3.35,37.25,51.25,2.054555e+05,2007.00
50%,3.76,50.50,60.00,1.270568e+06,2012.00
75%,4.21,65.00,77.00,1.106681e+07,2017.00
max,6.36,92.00,96.00,1.097979e+08,2026.00


## 4. Language and Script Distribution

In [ ]:
language_counts = (
    df["song_title_language"]
    .fillna("Unknown")
    .value_counts()
    .rename_axis("song_title_language")
    .reset_index(name="track_count")
)

language_counts["pct"] = language_counts["track_count"] / language_counts["track_count"].sum() * 100
language_counts

,song_title_language,track_count,pct
0,Latin-script title,156,65.546218
1,Chinese-script title,72,30.252101
2,Mixed-script title,9,3.781513
3,Japanese-script title,1,0.420168


In [ ]:
alt.Chart(language_counts).mark_bar().encode(
    x=alt.X("track_count:Q", title="Number of tracks"),
    y=alt.Y("song_title_language:N", sort="-x", title="Title language/script"),
    tooltip=[
        alt.Tooltip("song_title_language:N", title="Language/script"),
        alt.Tooltip("track_count:Q", title="Tracks"),
        alt.Tooltip("pct:Q", title="Share (%)", format=".1f"),
    ],
    color=alt.Color("song_title_language:N", legend=None),
).properties(
    title="Song Title Language Distribution",
    width=650,
    height=220,
)

alt.Chart(...)

In [ ]:
language_popularity = (
    df.groupby("song_title_language", dropna=False)
    .agg(
        track_count=("track_id", "count"),
        avg_popularity=("popularity", "mean"),
        median_popularity=("popularity", "median"),
        avg_artist_popularity=("artist_popularity", "mean"),
    )
    .sort_values("track_count", ascending=False)
    .round(2)
)

language_popularity

,track_count,avg_popularity,median_popularity,avg_artist_popularity
song_title_language,,,,
Latin-script title,156,52.72,54.0,65.17
Chinese-script title,72,44.46,47.5,54.92
Mixed-script title,9,43.44,43.0,47.67
Japanese-script title,1,7.00,7.0,57.00


In [ ]:
alt.Chart(df).mark_boxplot(size=45).encode(
    x=alt.X("song_title_language:N", title="Title language/script"),
    y=alt.Y("popularity:Q", title="Track popularity"),
    color=alt.Color("song_title_language:N", legend=None),
    tooltip=["song_title_language:N", "popularity:Q"],
).properties(
    title="Popularity Distribution by Title Language",
    width=650,
    height=320,
)

alt.Chart(...)

## 5. Genre Exploration

In [ ]:
def parse_list_like(value):
    """Parse Spotify list columns saved as strings."""
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []


genre_long = (
    df[["track_id", "track_name", "main_artist_name", "main_artist_genres", "popularity", "song_title_language"]]
    .assign(genre=lambda x: x["main_artist_genres"].apply(parse_list_like))
    .explode("genre")
    .dropna(subset=["genre"])
)

genre_long.head()

,track_id,track_name,main_artist_name,main_artist_genres,popularity,song_title_language,genre
0,0VGPId3riSeBV5FTRn850F,需要人陪,Leehom Wang,"['mandopop', 'c-pop', 'taiwanese pop', 'chinese r&b', 'malay pop']",50,Chinese-script title,mandopop
0,0VGPId3riSeBV5FTRn850F,需要人陪,Leehom Wang,"['mandopop', 'c-pop', 'taiwanese pop', 'chinese r&b', 'malay pop']",50,Chinese-script title,c-pop
0,0VGPId3riSeBV5FTRn850F,需要人陪,Leehom Wang,"['mandopop', 'c-pop', 'taiwanese pop', 'chinese r&b', 'malay pop']",50,Chinese-script title,taiwanese pop
0,0VGPId3riSeBV5FTRn850F,需要人陪,Leehom Wang,"['mandopop', 'c-pop', 'taiwanese pop', 'chinese r&b', 'malay pop']",50,Chinese-script title,chinese r&b
0,0VGPId3riSeBV5FTRn850F,需要人陪,Leehom Wang,"['mandopop', 'c-pop', 'taiwanese pop', 'chinese r&b', 'malay pop']",50,Chinese-script title,malay pop


In [ ]:
top_genres = (
    genre_long["genre"]
    .value_counts()
    .head(20)
    .rename_axis("genre")
    .reset_index(name="track_count")
)

top_genres

,genre,track_count
0,c-pop,85
1,mandopop,84
2,taiwanese pop,72
3,malay pop,59
4,chinese r&b,32
5,nu metal,18
6,rap metal,18
7,rock,18
8,alternative metal,18
9,gufeng,14


In [ ]:
alt.Chart(top_genres).mark_bar().encode(
    x=alt.X("track_count:Q", title="Number of tracks"),
    y=alt.Y("genre:N", sort="-x", title="Genre"),
    tooltip=["genre:N", "track_count:Q"],
).properties(
    title="Top Artist Genres in Playlist",
    width=650,
    height=430,
)

alt.Chart(...)

In [ ]:
genre_language = (
    genre_long[genre_long["genre"].isin(top_genres["genre"].head(12))]
    .groupby(["genre", "song_title_language"])
    .size()
    .reset_index(name="track_count")
)

alt.Chart(genre_language).mark_rect().encode(
    x=alt.X("song_title_language:N", title="Title language/script"),
    y=alt.Y("genre:N", sort="-x", title="Genre"),
    color=alt.Color("track_count:Q", title="Tracks"),
    tooltip=["genre:N", "song_title_language:N", "track_count:Q"],
).properties(
    title="Top Genres by Title Language",
    width=650,
    height=360,
)

alt.Chart(...)

In [ ]:
genre_tag_long = (
    df[["track_id", "track_name", "main_artist_name", "genre_language_tags", "popularity"]]
    .assign(genre_language_tag=lambda x: x["genre_language_tags"].apply(parse_list_like))
    .explode("genre_language_tag")
    .dropna(subset=["genre_language_tag"])
)

tag_summary = (
    genre_tag_long.groupby("genre_language_tag")
    .agg(
        track_count=("track_id", "count"),
        avg_popularity=("popularity", "mean"),
    )
    .sort_values("track_count", ascending=False)
    .round(2)
)

tag_summary

,track_count,avg_popularity
genre_language_tag,,
Chinese-associated,87,43.43
Western/Latin-title-associated,78,51.09
Western-associated,51,58.10
Korean-associated,13,54.69
Japanese-associated,11,42.36


## 6. Artist Patterns

In [ ]:
top_artists = (
    df.groupby("main_artist_name")
    .agg(
        track_count=("track_id", "count"),
        avg_track_popularity=("popularity", "mean"),
        artist_popularity=("artist_popularity", "max"),
        artist_followers=("artist_followers", "max"),
    )
    .sort_values(["track_count", "avg_track_popularity"], ascending=False)
    .head(20)
    .reset_index()
)

top_artists

,main_artist_name,track_count,avg_track_popularity,artist_popularity,artist_followers
0,Linkin Park,18,67.055556,89,34542635
1,Leehom Wang,15,51.400000,57,1192346
2,JJ Lin,9,44.444444,67,4257974
3,Avril Lavigne,7,56.571429,77,12579741
4,Justin Bieber,6,69.000000,96,91076968
5,BIGBANG,6,59.666667,71,6530949
6,David Tao,6,48.333333,60,1270568
7,F.I.R.,6,47.333333,56,264978
8,Jay Chou,5,59.000000,74,8429851
9,Maroon 5,5,57.600000,86,47831351


In [ ]:
alt.Chart(top_artists).mark_bar().encode(
    x=alt.X("track_count:Q", title="Number of tracks"),
    y=alt.Y("main_artist_name:N", sort="-x", title="Artist"),
    color=alt.Color("avg_track_popularity:Q", title="Avg track popularity"),
    tooltip=[
        "main_artist_name:N",
        "track_count:Q",
        alt.Tooltip("avg_track_popularity:Q", format=".1f"),
        "artist_popularity:Q",
        alt.Tooltip("artist_followers:Q", format=","),
    ],
).properties(
    title="Most Frequent Main Artists",
    width=650,
    height=430,
)

alt.Chart(...)

## 7. Popularity, Duration, and Release Year

In [ ]:
popularity_hist = alt.Chart(df).mark_bar().encode(
    x=alt.X(
        "popularity:Q",
        bin=alt.Bin(step=5),
        title="Track popularity",
    ),
    y=alt.Y("count():Q", title="Number of tracks", stack=None),
    tooltip=[
        alt.Tooltip("popularity:Q", bin=alt.Bin(step=5), title="Popularity range"),
        alt.Tooltip("count():Q", title="Tracks"),
    ],
).properties(
    title="Popularity Distribution",
    width=310,
    height=250,
)

duration_hist = alt.Chart(df).mark_bar().encode(
    x=alt.X(
        "duration_min:Q",
        bin=alt.Bin(step=0.25),
        title="Duration (minutes)",
    ),
    y=alt.Y("count():Q", title="Number of tracks", stack=None),
    tooltip=[
        alt.Tooltip("duration_min:Q", bin=alt.Bin(step=0.25), title="Duration range"),
        alt.Tooltip("count():Q", title="Tracks"),
    ],
).properties(
    title="Duration Distribution",
    width=310,
    height=250,
)

popularity_hist | duration_hist

alt.HConcatChart(...)

In [ ]:
alt.Chart(df).mark_circle(size=70, opacity=0.72).encode(
    x=alt.X("release_year:Q", title="Release year", scale=alt.Scale(zero=False)),
    y=alt.Y("popularity:Q", title="Track popularity"),
    color=alt.Color("song_title_language:N", title="Title language/script"),
    tooltip=["track_name:N", "main_artist_name:N", "release_year:Q", "popularity:Q", "song_title_language:N"],
).properties(
    title="Track Popularity by Release Year",
    width=650,
    height=360,
)

alt.Chart(...)

In [ ]:
release_decade = (
    df.dropna(subset=["release_year"])
    .assign(release_decade=lambda x: (x["release_year"] // 10 * 10).astype(int).astype(str) + "s")
)

decade_order = sorted(release_decade["release_decade"].unique())

decade_counts = (
    release_decade["release_decade"]
    .value_counts()
    .rename_axis("release_decade")
    .reset_index(name="track_count")
)

decade_bar = alt.Chart(decade_counts).mark_bar().encode(
    x=alt.X("release_decade:N", title="Release decade", sort=decade_order),
    y=alt.Y("track_count:Q", title="Number of tracks"),
    tooltip=["release_decade:N", "track_count:Q"],
).properties(
    title="Track Count by Release Decade",
    width=650,
    height=260,
)

decade_bar

alt.Chart(...)

In [ ]:
decade_language = (
    release_decade.groupby(["release_decade", "song_title_language"])
    .size()
    .reset_index(name="track_count")
)

alt.Chart(decade_language).mark_rect().encode(
    x=alt.X("release_decade:N", title="Release decade", sort=decade_order),
    y=alt.Y("song_title_language:N", title="Title language/script"),
    color=alt.Color("track_count:Q", title="Tracks"),
    tooltip=["release_decade:N", "song_title_language:N", "track_count:Q"],
).properties(
    title="Title Language by Release Decade",
    width=650,
    height=220,
)

alt.Chart(...)

## 8. Playlist Add Timeline

In [ ]:
added_timeline = (
    df.dropna(subset=["added_at"])
    .set_index("added_at")
    .resample("D")
    .size()
    .reset_index(name="tracks_added")
)

alt.Chart(added_timeline).mark_bar().encode(
    x=alt.X("added_at:T", title="Date added"),
    y=alt.Y("tracks_added:Q", title="Tracks added"),
    tooltip=["added_at:T", "tracks_added:Q"],
).properties(
    title="Tracks Added Over Time",
    width=650,
    height=260,
)

alt.Chart(...)

## 9. EDA Takeaways

### 9.1 Overall Playlist Profile

This playlist contains a relatively small but rich sample of tracks. The data includes track-level information, artist-level metadata, language/script features, genre tags, release dates, and playlist-added timestamps. These variables give enough structure for both descriptive visualization and later modelling.

The playlist is clearly multilingual and cross-cultural. Latin-script titles make up the largest group, while Chinese-script titles also represent a substantial part of the playlist. There are also smaller mixed-script and Japanese-script groups. Because some groups are much smaller than others, later statistical tests and models should treat language/script comparisons carefully.

### 9.2 Language and Popularity Patterns

The title-language distribution is one of the most important descriptive features in this project. It gives a quick view of the cultural and linguistic mix of the playlist. However, this feature is currently based on title script/language rather than confirmed sung language. For example, a song with a Latin-script title may not always be an English-language song. In the future, this could be improved with lyric-language detection or more detailed metadata.

The popularity-by-language visualization suggests that popularity may differ across language/script groups. This is useful as an exploratory finding, but it should not be interpreted too strongly from EDA alone. Spotify popularity reflects Spotify's user base and platform ecosystem, so it may underrepresent music scenes whose listeners primarily use other platforms.

### 9.3 Genre and Artist Patterns

The top-genre chart shows that the playlist has recognizable genre structure rather than being completely random. This is important for later recommendation modelling because genre can act as a useful content-based feature.

The most frequent main artists chart shows whether the playlist is concentrated around a few favorite artists or spread across many artists. This matters because a playlist dominated by a few artists may make recommendation models overfit to artist identity rather than broader musical taste. Artist popularity and followers are useful features, but they should be balanced with genre, language/script, and eventually audio or similarity features.

### 9.4 Release Era and Playlist Activity

The release-decade chart describes the time range of the music taste represented in the playlist. It helps answer whether the playlist leans toward older music, recent releases, or a mix across decades. In the later statistical inference notebook, release decade does not appear to provide strong evidence of popularity differences, but it remains useful as a descriptive feature for user profiling.

The tracks-added-over-time chart captures playlist-building behavior. It shows when tracks were added and whether additions happened gradually or in bursts. This is more of a user-behavior signal than a music-content signal. In a future website, this could become a timeline or calendar-style visualization showing how the user's music collection evolved.

### 9.5 Visualizations to Carry Forward

The strongest candidates for the web dashboard are:

- Song Title Language Distribution
- Popularity Distribution by Title Language
- Top Artist Genres in Playlist
- Most Frequent Main Artists
- Track Count by Release Decade
- Tracks Added Over Time

These charts are understandable and analytically useful, but many of them are bar charts. For the final website, they can serve as reliable baseline views, while selected charts can later be redesigned with more engaging visualization forms such as treemaps, bubble charts, calendar heatmaps, timeline views, or interactive artist/genre cards.

### 9.6 Implications for Next Steps

The EDA suggests that language/script, genre, artist identity, artist popularity, and release timing are all meaningful dimensions of the playlist. These features should be explored further in statistical inference and may become useful inputs for ML modelling.

At the same time, EDA also reveals limitations: the dataset is small, personally selected, culturally specific, and platform-dependent. Therefore, later modelling should focus less on universal claims about music popularity and more on explaining this user's listening profile and building personalized recommendations.